In [9]:
# ==============================================================================
# LAB 1: BÀI THỰC HÀNH THAO TÁC DỮ LIỆU
# Mã nguồn được chạy trên môi trường Jupyter Notebook của VS Code
# ==============================================================================

import pandas as pd
import numpy as np

# --- CÂU 3: Tải dữ liệu và in 10 dòng đầu, 10 dòng cuối ---
# (Đảm bảo file 'dulieuxettuyendaihoc.csv' nằm cùng thư mục với file notebook này)
df = pd.read_csv('dulieuxettuyendaihoc.csv')

print("=== 10 DÒNG ĐẦU TIÊN CỦA TẬP DỮ LIỆU ===")
display(df.head(10))

print("\n=== 10 DÒNG CUỐI CÙNG CỦA TẬP DỮ LIỆU ===")
display(df.tail(10))


# --- CÂU 4: Xử lý dữ liệu thiếu cho cột Dân tộc (DT) ---
print("\n=== KHẢO SÁT DỮ LIỆU THIẾU CỘT DT ===")
print("Các giá trị riêng biệt trước xử lý:", df['DT'].unique())
print("Số lượng dữ liệu thiếu cột DT:", df['DT'].isnull().sum())

# Thay thế dữ liệu thiếu bằng phương pháp điền giá trị 0
df['DT'] = df['DT'].fillna(0)
print("Số lượng dữ liệu thiếu cột DT sau khi xử lý:", df['DT'].isnull().sum())


# --- CÂU 5: Xử lý dữ liệu thiếu cho cột Toán lớp 10 (T1) bằng Mean ---
print("\n=== KHẢO SÁT DỮ LIỆU THIẾU CỘT T1 ===")
print("Số lượng dữ liệu thiếu cột T1:", df['T1'].isnull().sum())

# Tính giá trị trung bình (Mean) và điền vào chỗ trống
mean_t1 = df['T1'].mean()
df['T1'] = df['T1'].fillna(mean_t1)
print(f"Giá trị Mean dùng để thay thế cho cột T1: {mean_t1:.4f}")
print("Số lượng dữ liệu thiếu cột T1 sau khi xử lý:", df['T1'].isnull().sum())


# --- CÂU 6: Xử lý dữ liệu thiếu bằng Mean cho tất cả các biến điểm số còn lại ---
# Đồng bộ tên cột lỗi nếu 'HI' bị viết nhầm thay cho 'H1' (Hóa lớp 10)
if 'HI' in df.columns:
    df.rename(columns={'HI': 'H1'}, inplace=True)

# Danh sách tất cả các cột điểm số cần kiểm tra và xử lý thiếu bằng Mean
score_cols = [
    'T1', 'L1', 'H1', 'S1', 'V1', 'X1', 'D1', 'N1',
    'T2', 'L2', 'H2', 'S2', 'V2', 'X2', 'D2', 'N2',
    'T6', 'L6', 'H6', 'S6', 'V6', 'X6', 'D6', 'N6',
    'DH1', 'DH2', 'DH3'
]

for col in score_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())
print("\n[Đã hoàn thành] Xử lý dữ liệu thiếu bằng phương pháp Mean cho toàn bộ cột điểm số còn lại.")


# --- CÂU 7: Tạo các biến Trung bình môn TBM1, TBM2, TBM3 ---
# Công thức: TBM = (T*2 + L + H + S + V*2 + X + D + N) / 10
df['TBM1'] = (df['T1']*2 + df['L1'] + df['H1'] + df['S1'] + df['V1']*2 + df['X1'] + df['D1'] + df['N1']) / 10
df['TBM2'] = (df['T2']*2 + df['L2'] + df['H2'] + df['S2'] + df['V2']*2 + df['X2'] + df['D2'] + df['N2']) / 10
df['TBM3'] = (df['T6']*2 + df['L6'] + df['H6'] + df['S6'] + df['V6']*2 + df['X6'] + df['D6'] + df['N6']) / 10


# --- CÂU 8: Tạo các biến xếp loại học lực XL1, XL2, XL3 ---
def xep_loai(tbm):
    if tbm < 5.0:
        return 'Y'
    elif tbm < 6.5:
        return 'TB'
    elif tbm < 8.0:
        return 'K'
    elif tbm < 9.0:
        return 'G'
    else:
        return 'XS'

df['XL1'] = df['TBM1'].apply(xep_loai)
df['XL2'] = df['TBM2'].apply(xep_loai)
df['XL3'] = df['TBM3'].apply(xep_loai)


# --- CÂU 9: Quy đổi điểm sang thang điểm 4 của Mỹ (Min-Max Normalization) ---
# Thang điểm VN từ 0 -> 10, Thang Mỹ từ 0 -> 4. Áp dụng chuẩn hóa: US_TBM = (TBM / 10) * 4
df['US_TBM1'] = (df['TBM1'] / 10) * 4
df['US_TBM2'] = (df['TBM2'] / 10) * 4
df['US_TBM3'] = (df['TBM3'] / 10) * 4


# --- CÂU 10: Tạo biến Kết quả xét tuyển (KQXT) Đậu (1) hoặc Rớt (0) ---
def tinh_kqxt(row):
    khoi_thi = row['KT']
    dh1, dh2, dh3 = row['DH1'], row['DH2'], row['DH3']
    
    if khoi_thi in ['A', 'A1']:
        diem_xet = (dh1*2 + dh2 + dh3) / 4
        return 1 if diem_xet >= 5.0 else 0
    elif khoi_thi == 'B':
        diem_xet = (dh1 + dh2*2 + dh3) / 4
        return 1 if diem_xet >= 5.0 else 0
    else: # Khối C hoặc các khối khác
        diem_xet = (dh1 + dh2 + dh3) / 3
        return 1 if diem_xet >= 5.0 else 0

df['KQXT'] = df.apply(tinh_kqxt, axis=1)


# --- CÂU 11: Lưu kết quả xuống đĩa ---
df.to_csv('processed_dulieuxettuyendaihoc.csv', index=False)

=== 10 DÒNG ĐẦU TIÊN CỦA TẬP DỮ LIỆU ===


,STT,T1,L1,H1,S1,V1,X1,D1,N1,T2,...,X6,D6,N6,GT,DT,KV,DH1,DH2,DH3,KT
0,1,7.2,7.3,6.3,7.3,7.0,7.9,7.3,5.5,8.4,...,6.6,7.6,5.9,F,NaN,2NT,3.25,3.25,4.50,A1
1,2,5.4,3.9,3.9,4.0,5.4,5.4,5.3,2.8,6.3,...,6.6,6.1,4.4,M,NaN,1,6.00,4.00,3.50,C
2,3,5.6,6.8,7.2,7.5,4.3,7.4,5.8,3.2,5.0,...,7.9,8.1,4.6,M,NaN,1,5.00,6.75,4.00,C
3,4,6.6,6.4,5.3,6.9,5.4,7.3,6.4,5.8,5.1,...,7.1,7.3,7.4,M,NaN,1,4.25,4.25,5.25,D1
4,5,6.0,5.0,6.0,7.3,6.5,7.7,7.9,6.1,5.4,...,6.1,7.5,7.2,M,NaN,2NT,4.25,4.50,5.00,A
5,6,9.3,7.6,7.9,8.6,7.0,7.3,7.7,7.9,9.6,...,5.7,8.0,7.8,M,NaN,1,1.50,4.00,6.00,D1
6,7,2.8,3.9,5.5,6.9,5.0,7.3,4.6,5.2,4.4,...,6.6,6.0,6.0,F,NaN,2,6.50,6.75,5.25,C
7,8,8.3,6.0,7.6,5.1,7.5,4.7,5.8,7.2,6.7,...,7.1,6.8,7.0,F,NaN,2,3.75,4.50,4.25,D1
8,9,6.5,6.3,7.6,6.0,5.5,7.1,6.3,5.0,7.3,...,9.1,7.9,6.1,F,NaN,1,3.50,3.50,6.75,D1
9,10,7.3,5.9,4.7,7.1,6.7,7.9,6.7,7.7,8.0,...,6.4,6.1,7.8,F,NaN,1,4.00,4.75,5.50,D1



=== 10 DÒNG CUỐI CÙNG CỦA TẬP DỮ LIỆU ===


,STT,T1,L1,H1,S1,V1,X1,D1,N1,T2,...,X6,D6,N6,GT,DT,KV,DH1,DH2,DH3,KT
90,91,8.1,7.7,9.1,8.5,6.1,8.6,8.8,7.3,8.8,...,8.3,8.7,7.8,M,NaN,2,6.25,4.00,6.50,A
91,92,7.8,6.5,6.7,5.4,6.2,4.8,5.9,4.4,8.9,...,7.2,7.4,7.0,M,NaN,2NT,4.75,4.75,4.50,A
92,93,5.0,6.6,6.5,7.2,5.8,6.7,6.4,6.0,6.8,...,5.9,6.7,5.8,M,NaN,1,3.25,5.25,4.25,A
93,94,5.2,5.2,6.8,7.9,6.6,8.9,7.6,5.3,6.8,...,8.7,7.8,5.0,M,NaN,1,3.50,4.25,5.00,A
94,95,5.8,5.9,7.6,6.1,5.3,8.1,6.1,5.0,6.4,...,7.3,6.8,5.3,M,NaN,1,4.25,2.50,4.75,A
95,96,8.6,6.9,7.4,8.8,7.6,5.8,7.3,5.7,8.9,...,6.3,6.1,6.2,F,NaN,1,5.25,1.50,6.25,C
96,97,3.7,5.4,6.0,5.1,5.5,3.9,6.1,4.4,4.1,...,7.9,7.5,4.4,F,NaN,1,5.25,3.75,4.75,C
97,98,8.8,5.5,7.4,7.7,6.2,7.3,8.1,4.5,9.5,...,9.6,8.4,5.8,M,NaN,2NT,7.00,8.00,4.00,C
98,99,2.7,1.8,3.4,5.3,4.5,7.9,4.9,3.8,2.8,...,6.6,5.2,5.9,M,NaN,1,5.00,3.50,5.50,C
99,100,4.1,5.2,4.9,5.3,5.5,5.4,7.2,5.4,4.4,...,5.6,6.6,5.8,M,NaN,2NT,5.25,2.50,4.25,C



=== KHẢO SÁT DỮ LIỆU THIẾU CỘT DT ===
Các giá trị riêng biệt trước xử lý: [nan  1.  6.]
Số lượng dữ liệu thiếu cột DT: 97
Số lượng dữ liệu thiếu cột DT sau khi xử lý: 0

=== KHẢO SÁT DỮ LIỆU THIẾU CỘT T1 ===
Số lượng dữ liệu thiếu cột T1: 0
Giá trị Mean dùng để thay thế cho cột T1: 5.9460
Số lượng dữ liệu thiếu cột T1 sau khi xử lý: 0

[Đã hoàn thành] Xử lý dữ liệu thiếu bằng phương pháp Mean cho toàn bộ cột điểm số còn lại.
